# Topic Modeling of Recent Academic Publications using Top2Vec
## Discovering Hot Research Topics from 2020 to 2025

In [ ]:
!pip install top2vec

In [ ]:
!pip install numpy==1.26.4 --force-reinstall

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

# 1. Dataset

In [ ]:
path = "/content/drive/MyDrive/MachineLearning/processed.csv"
df = pd.read_csv(path)
df = df.dropna(subset=['cleaned_abstract'])
documents = df['cleaned_abstract'].tolist()
years = df["year"].astype(str).tolist()

print(f"Total documents: {len(documents)}")
print("First document sample:")
print(documents[0][:300], "...")

## 2. Modeling

### 2.1. Top2Vec Model

In [ ]:
from top2vec import Top2Vec
import time

start = time.time()

topic_model = Top2Vec(
    documents,
    embedding_model="all-MiniLM-L6-v2",
    speed="deep-learn",
    ngram_vocab=True,
    workers=4,
)

end = time.time()
print(f"Training completed in {end - start:.2f} seconds.")

In [ ]:
print("Topic count:", topic_model.get_num_topics())

In [ ]:
topic_sizes, topic_ids = topic_model.get_topic_sizes()

min_size = topic_sizes.min()
max_size = topic_sizes.max()

min_topic_id = topic_ids[topic_sizes.argmin()]
max_topic_id = topic_ids[topic_sizes.argmax()]

print(f"Smallest Topic {min_topic_id}: {min_size} docs")
print(f"Largest Topic  {max_topic_id}: {max_size} docs")

In [ ]:
topic_words, topic_scores, topic_ids = topic_model.get_topics()

print("First 5 Topics:")
for i in range(5):
    topic_id = topic_ids[i]
    words = topic_words[i]
    keywords = ", ".join(words[:10])
    print(f"Topic {topic_id}: {keywords}")

print("\nLast 5 Topics:")
for i in range(-5, 0):
    topic_id = topic_ids[i]
    words = topic_words[i]
    keywords = ", ".join(words[:10])
    print(f"Topic {topic_id}: {keywords}")

## 3. Visualizations

In [ ]:
from matplotlib import pyplot as plt

for topic_num in range(5):
    print(f"Topic {topic_num}")
    topic_model.generate_topic_wordcloud(topic_num)
    plt.show()

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt

topic_vecs = topic_model.topic_vectors
topic_words, _, topic_ids = topic_model.get_topics()

labels = [
    f"{tid} | {', '.join(words[:3])}" for tid, words in zip(topic_ids, topic_words)
]

dist_matrix = cosine_distances(topic_vecs)
condensed_dist = squareform(dist_matrix, checks=False)
linkage_matrix = linkage(condensed_dist, method='average')

plt.figure(figsize=(12, 20))
dendrogram(
    linkage_matrix,
    orientation='right',
    labels=labels,
    leaf_font_size=5,
)
plt.title("Top2Vec Topic Hierarchy")
plt.xlabel("Distance")
plt.ylabel("Topics")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt

sim_matrix = cosine_similarity(topic_vecs)

plt.figure(figsize=(10, 8))
sns.heatmap(
    sim_matrix,
    xticklabels=False,
    yticklabels=False,
    cmap='coolwarm',
    square=True,
    cbar=True
)

plt.title("Topic-Topic Cosine Similarity Heatmap", fontsize=14)
plt.tight_layout()
plt.show()

### Topics over Time - Growing Topics from 2020 to 2025

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

doc_vecs = topic_model.document_vectors
topic_vecs = topic_model.topic_vectors

dominant_topic_ids = []
for vec in tqdm(doc_vecs):
    sims = cosine_similarity([vec], topic_vecs)[0]
    best_topic = np.argmax(sims)
    dominant_topic_ids.append(int(best_topic))

df_topics = pd.DataFrame({
    "Year": years,
    "Topic": dominant_topic_ids
})

topic_counts = df_topics.groupby(["Year", "Topic"]).size().reset_index(name="Frequency")
pivot = topic_counts.pivot_table(index="Year", columns="Topic", values="Frequency", fill_value=0)

first_year = pivot.index.min()
last_year = pivot.index.max()

growth = pivot.loc[last_year] - pivot.loc[first_year]
growth = growth[growth > 0]
top_growing_topics = growth.sort_values(ascending=False).head(10)

topic_words, _, topic_ids = topic_model.get_topics()
id_to_words = {tid: words for tid, words in zip(topic_ids, topic_words)}

print(f"\nGrowing Topics from {first_year} to {last_year}\n")
for tid in top_growing_topics.index:
    keywords = ", ".join(id_to_words.get(tid, [])[:10])
    print(f"Topic {tid}: {keywords}")

#### Growing Topics from 2020 to 2025

| Topic ID | Description                                                              |
|----------|--------------------------------------------------------------------------|
| 18       | Reasoning abilities and chains in language models                        |
| 14       | Photorealistic and volumetric rendering techniques                       |
| 56       | Agentic workflows and collaborative AI agents                            |
| 31       | Diffusion-based generative models and video synthesis                    |
| 101      | Multimodal reasoning and cross-modal retrieval                           |
| 36       | Large language models and multilingual pretraining                       |
| 74       | Text-to-image generation and contrastive vision-language models          |
| 35       | Code generation and programming assistant tools                          |
| 127      | Linguistic cues and human-like reasoning in conversational AI            |
| 73       | Automatic evaluation in NLG and annotation quality assessment            |


In [ ]:
import matplotlib.pyplot as plt

top_topics = top_growing_topics.index[:10]

topic_words, _, topic_ids = topic_model.get_topics()
id_to_words = {tid: words for tid, words in zip(topic_ids, topic_words)}

plt.figure(figsize=(12, 6))

for tid in top_topics:
    label = f"Topic {tid}: {', '.join(id_to_words.get(tid, [])[:3])}"
    plt.plot(pivot.index, pivot[tid], marker='o', label=label)

plt.title("En Hızlı Büyüyen Araştırma Konuları", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Document Count")
plt.legend(fontsize=9)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
growing_topic_ids = [1, 29, 28, 31, 7]

for topic_id in growing_topic_ids:
    print(f"Topic {topic_id}")
    topic_model.generate_topic_wordcloud(topic_id)
    plt.show()

In [ ]:
import umap
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

umap_model = umap.UMAP(n_components=2, random_state=42)
doc_vecs_2d = umap_model.fit_transform(topic_model.document_vectors)

topics = np.array(dominant_topic_ids)

plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=doc_vecs_2d[:, 0],
    y=doc_vecs_2d[:, 1],
    hue=topics,
    palette="tab10",
    legend=False,
    s=10,
    alpha=0.7
)

plt.title("Top2Vec Document Clusters via UMAP")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()
plt.show()

## 4. Performance Metrics

### 4.1. Coherence Scores

In [ ]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

tokenized_docs = [doc.split() for doc in documents if isinstance(doc, str)]

dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

topics_words = []

for topic in topic_words:
    clean_words = []
    for word in topic[:10]:
        if isinstance(word, str):
            if " " not in word:
                clean_words.append(word)
            else:
                clean_words.extend(word.split())
    if clean_words:
        topics_words.append(clean_words)

coherence_cv = CoherenceModel(topics=topics_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_v').get_coherence()
coherence_umass = CoherenceModel(topics=topics_words, texts=tokenized_docs, dictionary=dictionary, coherence='u_mass').get_coherence()
coherence_npmi = CoherenceModel(topics=topics_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_npmi').get_coherence()
coherence_uci = CoherenceModel(topics=topics_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_uci').get_coherence()

print(f"C_v Coherence:     {coherence_cv:.4f}")
print(f"U_Mass Coherence: {coherence_umass:.4f}")
print(f"NPMI Coherence:   {coherence_npmi:.4f}")
print(f"UCI Coherence:    {coherence_uci:.4f}")

### 4.2. PUW

In [ ]:
all_words = [word for topic in topics_words for word in topic]
unique_words = set(all_words)
puw = len(unique_words) / len(all_words)
print(f"PUW (Proportion of Unique Words): {puw:.4f}")

### 4.3. Avg. Jaccard Similarity

In [ ]:
from itertools import combinations

def jaccard_similarity(set1, set2):
    return len(set1 & set2) / len(set1 | set2)

jaccard_scores = []
for t1, t2 in combinations(topics_words, 2):
    jaccard_scores.append(jaccard_similarity(set(t1), set(t2)))

avg_jaccard = sum(jaccard_scores) / len(jaccard_scores)
print(f"Average Jaccard Similarity between topics: {avg_jaccard:.4f}")

In [ ]:
from sklearn.metrics.pairwise import cosine_distances
import numpy as np

topic_vecs = topic_model.topic_vectors

dists = cosine_distances(topic_vecs)

upper_triangle = dists[np.triu_indices_from(dists, k=1)]
cdm = upper_triangle.mean()

print(f"Cosine Distance Mean (CDM): {cdm:.4f}")

In [ ]:
min_dists = np.sort(dists + np.eye(len(dists)) * 2, axis=1)[:, 0]
edi = min_dists.mean()

print(f"Embedding Diversity Index (EDI): {edi:.4f}")